# 01. Data understanding, preprocessing and conversion
HW2: Multiple Object Video Detection (UCU AI 2026)

**Sections:** 0. Config, 1. Data structure, 2. EDA, 3. Conversion (COCO + YOLO), 4. Visual check of ground truth.

Run on CPU (no GPU needed). Output: `OUT` folder with `coco_train.json`, `coco_val.json`, `split.json` and a `yolo/` folder.
Save `OUT` as a Kaggle Dataset afterwards so the next notebooks can use it as an Input.

## 0. Config
Change only this cell. `ROOT` must be the folder that contains the sequences (frames) and the annotation txt files.

In [1]:
from pathlib import Path
import json, os, random, shutil

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

ROOT = Path("/kaggle/input/multiple-object-video-detection-hw-2-ucu-ai-2026")  # <-- SET ME
ANN_SUBDIR = "annotations"   # folder with one txt per sequence
SEQ_SUBDIR = "sequences"     # folder with one subfolder of frames per sequence
OUT = Path("/kaggle/working/hw2_data")

STEP = 4          # keep every STEP-th frame (frames are highly correlated, see section 3)
VAL_FRAC = 0.2    # share of SEQUENCES used for validation
SEED = 42
OTHERS = "drop"   # class 11 ("others"): "drop" or "ignore" (treat as ignored region)
COPY_IMAGES = True  # True: copy images (needed if OUT will be saved as a Kaggle Dataset). False: symlinks

## 1. Data structure
Check that the folders look as expected before anything else. If not, fix `ROOT`, `ANN_SUBDIR`, `SEQ_SUBDIR` above.

In [ ]:
def show_tree(path, depth=2, max_items=6, prefix=""):
    """Print the first levels of a folder."""
    items = sorted(Path(path).iterdir())
    for p in items[:max_items]:
        print(prefix + p.name + ("/" if p.is_dir() else ""))
        if p.is_dir() and depth > 1:
            show_tree(p, depth - 1, max_items, prefix + "    ")
    if len(items) > max_items:
        print(prefix + f"... (+{len(items) - max_items} more)")

show_tree(ROOT)

In [ ]:
# VisDrone-VID annotation line:
# frame_index, target_id, left, top, width, height, score, category, truncation, occlusion
# category: 0 = ignored region, 1..10 = target classes, 11 = others
CLASS_NAMES = {1: "pedestrian", 2: "people", 3: "bicycle", 4: "car", 5: "van",
               6: "truck", 7: "tricycle", 8: "awning-tricycle", 9: "bus", 10: "motor"}
IGNORED, OTHER_CLS = 0, 11
COLUMNS = ["frame", "track_id", "x", "y", "w", "h", "score", "cat", "trunc", "occl"]

def load_annotations(root):
    """All annotation files -> one DataFrame with a 'seq' column."""
    parts = []
    for txt in sorted((Path(root) / ANN_SUBDIR).glob("*.txt")):
        d = pd.read_csv(txt, header=None, names=COLUMNS, usecols=range(10))
        d["seq"] = txt.stem
        parts.append(d)
    if not parts:
        raise FileNotFoundError(f"No .txt files in {Path(root) / ANN_SUBDIR}")
    return pd.concat(parts, ignore_index=True)

def frame_path(root, seq, frame):
    return Path(root) / SEQ_SUBDIR / seq / f"{int(frame):07d}.jpg"

def size_bucket(w, h):
    a = w * h
    return "small" if a < 32 ** 2 else ("medium" if a < 96 ** 2 else "large")

df = load_annotations(ROOT)
df.head()

## 2. EDA
Goal: facts that justify later choices (input resolution, anchors, number of queries, augmentation).
Write the conclusions into `experiments_log.md`.

In [ ]:
ign = df[df.cat == IGNORED]
oth = df[df.cat == OTHER_CLS]
obj = df[df.cat.isin(CLASS_NAMES)].copy()
obj["bucket"] = [size_bucket(w, h) for w, h in zip(obj.w, obj.h)]
obj["area"] = obj.w * obj.h

print(f"Sequences: {df.seq.nunique()} | annotated frames: {df.groupby(['seq', 'frame']).ngroups}")
print(f"Target objects: {len(obj)} | ignored regions: {len(ign)} | 'others' (cls 11): {len(oth)}")

In [ ]:
# Class balance and object sizes
cls_counts = obj.cat.map(CLASS_NAMES).value_counts()
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
cls_counts.plot.bar(ax=ax[0], title="Objects per class (log)"); ax[0].set_yscale("log")
ax[1].hist(np.sqrt(obj.area), bins=80); ax[1].set_title("Box size, sqrt(area) in px")
ax[2].hist(obj.h / obj.w.clip(lower=1), bins=80, range=(0, 5)); ax[2].set_title("Aspect ratio h/w")
plt.tight_layout(); plt.show()

print("Share by COCO size bucket:\n", obj.bucket.value_counts(normalize=True).round(3).to_string())
print("\nSize bucket per class:\n", obj.groupby(obj.cat.map(CLASS_NAMES)).bucket.value_counts(normalize=True).unstack().round(2).to_string())
print(f"\nMedian box: {obj.w.median():.0f} x {obj.h.median():.0f} px")

In [ ]:
# Scene density and variation across sequences
per_frame = obj.groupby(["seq", "frame"]).size()
seq_tbl = pd.concat([obj.groupby("seq").size().rename("objects"),
                     df.groupby("seq").frame.nunique().rename("frames")], axis=1)
seq_tbl["obj_per_frame"] = (seq_tbl.objects / seq_tbl.frames).round(1)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
per_frame.plot.hist(bins=60, ax=ax[0], title="Objects per frame")
seq_tbl.obj_per_frame.plot.hist(bins=30, ax=ax[1], title="Mean objects per frame, per sequence")
plt.tight_layout(); plt.show()

print("Objects per frame: mean=%.1f median=%.0f p95=%.0f max=%d" % (
    per_frame.mean(), per_frame.median(), per_frame.quantile(0.95), per_frame.max()))
seq_tbl.sort_values("obj_per_frame").iloc[[0, -1]]  # sparsest and densest sequence

## 3. Conversion to COCO and YOLO
Decisions (explain them in the report):
- **Split by sequence**, not by frame: neighbouring frames are almost identical, a frame-level split would leak.
- **Frame step `STEP`**: consecutive frames are near-duplicates, so every STEP-th frame keeps most of the information and cuts epoch time.
- **Ignored regions**: COCO `iscrowd=1` (one entry per class, otherwise pycocotools does not skip detections inside the region because it matches by category); YOLO: region is painted grey in the image copy.
- **Class 11 "others"**: controlled by `OTHERS`.
- Classes 1..10 become YOLO ids 0..9 and COCO category ids 1..10.

In [ ]:
def split_sequences(seqs, val_frac, seed):
    seqs = sorted(seqs)
    random.Random(seed).shuffle(seqs)
    n_val = max(1, round(len(seqs) * val_frac))
    return set(seqs[n_val:]), set(seqs[:n_val])

def put_image(src, dst):
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        return
    if COPY_IMAGES:
        shutil.copy2(src, dst)
    else:
        os.symlink(Path(src).resolve(), dst)

def put_masked_image(src, dst, regions):
    dst.parent.mkdir(parents=True, exist_ok=True)
    img = Image.open(src).convert("RGB")
    for x, y, w, h in regions:
        img.paste((114, 114, 114), (int(x), int(y), int(x + w), int(y + h)))
    img.save(dst, quality=95)

In [ ]:
train_seqs, val_seqs = split_sequences(df.seq.unique(), VAL_FRAC, SEED)
OUT.mkdir(parents=True, exist_ok=True)
(OUT / "split.json").write_text(json.dumps(
    {"train": sorted(train_seqs), "val": sorted(val_seqs), "step": STEP, "seed": SEED}, indent=1))
print(f"train sequences: {len(train_seqs)} | val sequences: {len(val_seqs)}")

for split, seqs in (("train", train_seqs), ("val", val_seqs)):
    coco = {"images": [], "annotations": [],
            "categories": [{"id": c, "name": n} for c, n in CLASS_NAMES.items()]}
    ann_id, img_id = 1, 1
    sub = df[df.seq.isin(seqs)]
    for seq, g_seq in sub.groupby("seq"):
        for fr in sorted(g_seq.frame.unique())[::STEP]:
            g = g_seq[g_seq.frame == fr]
            src = frame_path(ROOT, seq, fr)
            if not src.exists():
                continue
            W, H = Image.open(src).size
            name = f"{seq}_{int(fr):07d}.jpg"
            ign_mask = (g.cat == IGNORED) | ((g.cat == OTHER_CLS) & (OTHERS == "ignore"))
            regions = list(zip(g[ign_mask].x, g[ign_mask].y, g[ign_mask].w, g[ign_mask].h))
            objs = g[g.cat.isin(CLASS_NAMES)]

            coco["images"].append({"id": img_id, "file_name": name, "width": W, "height": H,
                                   "seq": seq, "frame": int(fr)})
            for r in objs.itertuples():
                coco["annotations"].append({"id": ann_id, "image_id": img_id, "category_id": int(r.cat),
                                            "bbox": [float(r.x), float(r.y), float(r.w), float(r.h)],
                                            "area": float(r.w * r.h), "iscrowd": 0})
                ann_id += 1
            for (x, y, w, h) in regions:          # crowd region for every class
                for c in CLASS_NAMES:
                    coco["annotations"].append({"id": ann_id, "image_id": img_id, "category_id": c,
                                                "bbox": [float(x), float(y), float(w), float(h)],
                                                "area": float(w * h), "iscrowd": 1})
                    ann_id += 1

            dst = OUT / "yolo" / "images" / split / name
            put_masked_image(src, dst, regions) if regions else put_image(src, dst)
            lab = OUT / "yolo" / "labels" / split / (Path(name).stem + ".txt")
            lab.parent.mkdir(parents=True, exist_ok=True)
            with open(lab, "w") as f:
                for r in objs.itertuples():
                    cx, cy = (r.x + r.w / 2) / W, (r.y + r.h / 2) / H
                    f.write(f"{int(r.cat) - 1} {min(max(cx, 0), 1):.6f} {min(max(cy, 0), 1):.6f} "
                            f"{min(r.w / W, 1):.6f} {min(r.h / H, 1):.6f}\n")
            img_id += 1
    (OUT / f"coco_{split}.json").write_text(json.dumps(coco))
    n_obj = sum(a["iscrowd"] == 0 for a in coco["annotations"])
    print(f"{split}: {len(coco['images'])} frames, {n_obj} objects")

names = "\n".join(f"  {c - 1}: {n}" for c, n in CLASS_NAMES.items())
(OUT / "yolo" / "data.yaml").write_text(
    f"path: {(OUT / 'yolo').resolve()}\ntrain: images/train\nval: images/val\nnames:\n{names}\n")

## 4. Visual check of ground truth
Look at the picture before training anything. Green = objects, red dashed = ignored regions. Boxes must sit on real objects, in several different sequences.

In [ ]:
def show_gt(split="train", n=9, seed=0):
    coco = json.loads((OUT / f"coco_{split}.json").read_text())
    by_img = {}
    for a in coco["annotations"]:
        by_img.setdefault(a["image_id"], []).append(a)
    imgs = random.Random(seed).sample(coco["images"], min(n, len(coco["images"])))
    cols = 3
    rows = (len(imgs) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 4 * rows))
    axes = np.atleast_1d(axes).flatten()
    for ax, im in zip(axes, imgs):
        ax.imshow(Image.open(OUT / "yolo" / "images" / split / im["file_name"]))
        seen = set()
        for a in by_img.get(im["id"], []):
            x, y, w, h = a["bbox"]
            if a["iscrowd"]:
                if (x, y, w, h) not in seen:
                    seen.add((x, y, w, h))
                    ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, ec="red", lw=2, ls="--"))
            else:
                ax.add_patch(patches.Rectangle((x, y), w, h, fill=False, ec="lime", lw=1))
                ax.text(x, y, CLASS_NAMES[a["category_id"]], color="yellow", fontsize=6)
        ax.set_title(im["file_name"], fontsize=8); ax.axis("off")
    for ax in axes[len(imgs):]:
        ax.axis("off")
    plt.tight_layout(); plt.show()

show_gt("train")

In [ ]:
show_gt("val", n=6, seed=1)

## Next
1. Write EDA conclusions and conversion decisions into `experiments_log.md`.
2. Save `OUT` as a Kaggle Dataset (Output tab, *Save Version*, then *New Dataset*), or download it.
3. Continue in `02_yolo_baseline.ipynb`.